# ULog flight analysis

Reads a flight log written by this repository — or by PX4 — and works through
the questions you actually ask after a flight, in the order you ask them:

1. Did it fly the mission, and what did the state machine do?
2. How well did it track, and *where* did the error go?
3. Was the estimator healthy, or was the controller chasing a bad estimate?
4. Did anything trip, and why?

The ULog reader is `mission_control.logs.ulog`, which has no dependency on
`pyulog`. A log from a real PX4 vehicle opens with the same code; the topic
names were chosen to match PX4's so the plotting below works on either.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(ROOT, "src"))

from mission_control.logs.flight_log import STATE_NAMES
from mission_control.logs.ulog import UlogReader

plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
# Pick a log. Run `python analysis/run_comparison.py` first if results/ is empty.
LOG_DIR = os.path.join(ROOT, "results", "logs")
candidates = sorted(f for f in os.listdir(LOG_DIR) if f.endswith(".ulg")) \
    if os.path.isdir(LOG_DIR) else []

if not candidates:
    raise SystemExit(
        "No logs found. Run: python analysis/run_comparison.py --quick"
    )

path = os.path.join(LOG_DIR, candidates[0])
log = UlogReader(path)
print(candidates[0])
for name, count in log.summary()["topics"].items():
    print(f"  {name:40s} {count:6d} samples")
print()
print("info:", log.info)

## 1. What did the state machine do?

Before looking at any error, establish which phase the vehicle was in. A large
error during RTL means something completely different from the same error
during the survey.

In [ ]:
status = log.topics["vehicle_status"]
t = (status["timestamp"] - status["timestamp"][0]) * 1e-6
state = status["nav_state"].astype(int)

changes = np.flatnonzero(np.diff(state)) + 1
print(f"{'time (s)':>9}  phase")
for index in np.concatenate(([0], changes)):
    print(f"{t[index]:9.2f}  {STATE_NAMES.get(state[index], '?')}")

fig, ax = plt.subplots()
ax.step(t, state, where="post", linewidth=1.4)
ax.set_yticks(sorted(STATE_NAMES))
ax.set_yticklabels([STATE_NAMES[k] for k in sorted(STATE_NAMES)], fontsize=8)
ax.set_xlabel("time (s)")
ax.set_title("mission phase")
plt.show()

## 2. Tracking

Ground truth against the reference. Note that the *estimate* is a third series:
the difference between truth and reference is tracking error, and the difference
between estimate and truth is estimation error. Confusing them is the most
common way to misread a flight.

In [ ]:
truth = log.topics["vehicle_local_position_groundtruth"]
estimate = log.topics["vehicle_local_position"]
reference = log.topics["vehicle_local_position_setpoint"]

mission_phase = state == 3  # 'mission'

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
ax = axes[0]
ax.plot(reference["y"][mission_phase], reference["x"][mission_phase],
        color="#495057", linewidth=2.2, alpha=0.5, label="reference")
ax.plot(truth["y"][mission_phase], truth["x"][mission_phase],
        color="#E8590C", linewidth=1.0, label="flown")
ax.set_xlabel("east (m)"); ax.set_ylabel("north (m)")
ax.set_aspect("equal", adjustable="datalim"); ax.legend(frameon=False)
ax.set_title("ground track, mission phase")

error = np.column_stack([
    truth["x"] - reference["x"],
    truth["y"] - reference["y"],
    truth["z"] - reference["z"],
])
norm = np.linalg.norm(error, axis=1)

ax = axes[1]
ax.plot(t, norm, linewidth=0.9, color="#E8590C")
ax.axhline(np.sqrt(np.mean(norm[mission_phase] ** 2)), linestyle="--",
           color="#212529", linewidth=1.0, label="mission-phase RMS")
ax.set_xlabel("time (s)"); ax.set_ylabel("position error (m)")
ax.legend(frameon=False); ax.set_title("tracking error")
plt.tight_layout(); plt.show()

print(f"mission phase: RMS {np.sqrt(np.mean(norm[mission_phase]**2)):.4f} m, "
      f"max {norm[mission_phase].max():.4f} m")

### Where does the error go?

Splitting the error along and across the reference path matters more than the
total. Along-track error is a timing offset — the survey photo was taken a
metre early. Cross-track error is what puts an aircraft into a structure.

In [ ]:
velocity = np.column_stack([reference["vx"], reference["vy"], reference["vz"]])
speed = np.linalg.norm(velocity, axis=1)
moving = (speed > 0.2) & mission_phase

direction = np.zeros_like(velocity)
direction[moving] = velocity[moving] / speed[moving, None]
along = np.sum(error * direction, axis=1)
cross = np.linalg.norm(error - along[:, None] * direction, axis=1)

fig, ax = plt.subplots()
ax.plot(t[moving], cross[moving], linewidth=0.9, label="cross-track", color="#C92A2A")
ax.plot(t[moving], along[moving], linewidth=0.9, label="along-track (signed)",
        color="#1C7ED6")
ax.set_xlabel("time (s)"); ax.set_ylabel("error (m)")
ax.legend(frameon=False); ax.set_title("error decomposed")
plt.show()

print(f"cross-track RMS  {np.sqrt(np.mean(cross[moving]**2)):.4f} m")
print(f"along-track RMS  {np.sqrt(np.mean(along[moving]**2)):.4f} m  "
      f"(mean {along[moving].mean():+.4f} m; negative means lagging)")

## 3. Was the estimator healthy?

If it was not, none of the above means what it appears to. Two checks: the
actual error against truth, and whether the filter's own reported uncertainty
was telling the truth about it.

In [ ]:
estimator = log.topics["estimator_status"]
estimation_error = np.column_stack([
    estimate["x"] - truth["x"],
    estimate["y"] - truth["y"],
    estimate["z"] - truth["z"],
])

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
for index, (name, colour) in enumerate(
        (("north", "#1C7ED6"), ("east", "#37B24D"), ("down", "#F76707"))):
    axes[0].plot(t, estimation_error[:, index], linewidth=0.8, color=colour, label=name)
axes[0].plot(t, estimator["position_std_h"], "k--", linewidth=1.0,
             label="reported 1-sigma")
axes[0].plot(t, -estimator["position_std_h"], "k--", linewidth=1.0)
axes[0].set_ylabel("estimate - truth (m)"); axes[0].legend(frameon=False, ncol=4, fontsize=8)

axes[1].plot(t, estimator["gnss_fix_type"], linewidth=1.2, color="#5F3DC4")
axes[1].set_yticks([0, 3, 5, 6])
axes[1].set_yticklabels(["none", "3D", "RTK float", "RTK fixed"], fontsize=8)
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("GNSS fix")
plt.tight_layout(); plt.show()

nis = estimator["position_nis"]
usable = nis > 0
print(f"estimation RMS  {np.sqrt(np.mean(np.sum(estimation_error**2, axis=1))):.4f} m")
print(f"mean NIS        {nis[usable].mean():.2f}  (a consistent 3-DOF filter averages ~3)")
print("  below 3: the filter overstates its measurement noise and discards information")
print("  above 3: it is overconfident and will start rejecting good fixes")

## 4. What did the controller demand, and what did it cost?

In [ ]:
attitude = log.topics["vehicle_attitude_setpoint"]
tracker = log.topics["trajectory_tracker_status"]

tilt = np.degrees(np.arccos(np.clip(
    np.cos(attitude["roll_body"]) * np.cos(attitude["pitch_body"]), -1, 1)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(t, tilt, linewidth=0.8, color="#E8590C")
axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("commanded tilt (deg)")
axes[0].set_title("tilt demand")

axes[1].plot(t, attitude["thrust_body_2"], linewidth=0.8, color="#1C7ED6")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("normalised thrust")
axes[1].set_title("collective")

axes[2].plot(t, tracker["solve_time_s"] * 1e6, linewidth=0.6,
             color="#5F3DC4", label="optimisation")
axes[2].plot(t, tracker["update_time_s"] * 1e6, linewidth=0.6,
             color="#868E96", label="whole step")
axes[2].axhline(20000, color="#C92A2A", linestyle=":", label="50 Hz period")
axes[2].set_yscale("log"); axes[2].set_xlabel("time (s)")
axes[2].set_ylabel("microseconds"); axes[2].legend(frameon=False, fontsize=8)
axes[2].set_title("compute per control step")
plt.tight_layout(); plt.show()

saturated = tracker["saturated"].mean()
print(f"saturated for {100 * saturated:.1f}% of the flight")
print("  a high figure means the reference was asking for more than the vehicle has,")
print("  and the tracking error is partly a property of the trajectory, not the controller")

## 5. Safety

Geofence margin and battery, plus anything the failsafe matrix did.

In [ ]:
fence = log.topics["geofence_result"]
battery = log.topics["battery_status"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
margin = np.where(fence["margin_m"] > 1e5, np.nan, fence["margin_m"])
axes[0].plot(t, margin, linewidth=0.9, color="#0B7285")
axes[0].axhline(0.0, color="#C92A2A", linewidth=1.0)
axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("distance to boundary (m)")
axes[0].set_title("geofence margin (positive is inside)")

axes[1].plot(t, 100 * battery["remaining"], linewidth=1.0, color="#37B24D",
             label="state of charge (%)")
twin = axes[1].twinx()
twin.plot(t, battery["voltage_v"], linewidth=0.9, color="#F76707",
          label="terminal voltage (V)")
twin.set_ylabel("volts")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("percent")
axes[1].set_title("battery")
plt.tight_layout(); plt.show()

if np.any(fence["breached"] > 0.5):
    print(f"geofence breached at t = {t[fence['breached'] > 0.5][0]:.1f} s")
else:
    print(f"geofence never breached; closest approach {np.nanmin(margin):.2f} m")

failsafe = status["failsafe_action"].astype(int)
if failsafe.max() > 0:
    first = np.flatnonzero(failsafe > 0)[0]
    print(f"failsafe action {failsafe.max()} first raised at t = {t[first]:.1f} s")
else:
    print("no failsafe action during this flight")

## Opening a real PX4 log

The same reader takes a `.ulg` off an SD card. PX4 publishes
`vehicle_local_position`, `vehicle_local_position_setpoint` and
`vehicle_attitude_setpoint` under exactly these names, so sections 2 and 4 work
unchanged. `vehicle_local_position_groundtruth` only exists in SITL, so on a
real flight there is no truth to compare against — which is why the estimator
section becomes a consistency check on the innovations rather than an error
measurement.

```python
log = UlogReader("/path/to/log_1_2026-8-10-11-30-00.ulg")
```